In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rayyankauchali0/resume-dataset/resumes_dataset.jsonl
/kaggle/input/models/keras/bert/keras/bert_base_en_uncased/3/config.json
/kaggle/input/models/keras/bert/keras/bert_base_en_uncased/3/tokenizer.json
/kaggle/input/models/keras/bert/keras/bert_base_en_uncased/3/metadata.json
/kaggle/input/models/keras/bert/keras/bert_base_en_uncased/3/model.weights.h5
/kaggle/input/models/keras/bert/keras/bert_base_en_uncased/3/assets/tokenizer/vocabulary.txt


In [2]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import json
import keras
import keras_hub
import pandas as pd
import numpy as np

# 1. Load the data using your verified schema
file_path = "/kaggle/input/datasets/rayyankauchali0/resume-dataset/resumes_dataset.jsonl"

raw_records = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        raw_records.append(json.loads(line))

df = pd.DataFrame(raw_records)

# 2. Combine text sections into a single comprehensive resume profile string
df['full_resume_text'] = (
    df['Summary'].fillna('') + "\n" + 
    df['Skills'].fillna('') + "\n" + 
    df['Experience'].fillna('')
)

print(f"📊 Dataset Ingested! Successfully loaded {len(df)} professional tech records.")
print("\n--- Available Tech Categories ---")
print(df['Category'].value_counts().head(6))


📊 Dataset Ingested! Successfully loaded 3500 professional tech records.

--- Available Tech Categories ---
Category
Java Developer      200
Python Developer    200
Data Science        200
DevOps              180
SQL Developer       180
Database            150
Name: count, dtype: int64


In [3]:
# Slicing the data to focus on highly similar technical positions
target_roles = ["Backend Developer", "Frontend Developer", "Machine Learning Engineer", "Data Scientist"]
filtered_df = df[df['Category'].isin(target_roles)].reset_index(drop=True)

# Sample a clean batch of 20 resumes to run through our encoder
sample_df = filtered_df.sample(n=20, random_state=101).reset_index(drop=True)

test_resumes = sample_df['full_resume_text'].tolist()
all_candidate_labels = [f"Candidate {i+1} ({role})" for i, role in enumerate(sample_df['Category'])]

# Load the single 768-dimensional master BERT model
master_model = keras_hub.models.BertTextClassifier.from_preset(
    "bert_base_en_uncased", 
    num_classes=2
)
preprocessor = master_model.preprocessor
backbone = master_model.backbone

print(f"🎯 Test Pool Formed: Evaluating {len(test_resumes)} closely related tech candidates.")
print("🤖 Master Transformer core successfully loaded into memory.")


I0000 00:00:1785650165.841575      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785650165.844525      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


🎯 Test Pool Formed: Evaluating 20 closely related tech candidates.
🤖 Master Transformer core successfully loaded into memory.


In [11]:
# 1. Define the specific target vacancy description
target_job_description = (
    "We are seeking an Advanced Machine Learning Engineer. The primary responsibility "
    "is to deploy deep neural network models, configure distributed pipeline architectures, "
    "and perform hyperparameter tuning using TensorFlow, Keras, and PyTorch. "
    "Strong foundations in predictive modeling and natural language processing are mandatory."
)

# 2. Extract Transformer Embeddings
jd_inputs = preprocessor(np.array([target_job_description]))
jd_vector = backbone(jd_inputs)["pooled_output"].numpy()

resume_inputs = preprocessor(np.array(test_resumes))
resume_vectors = backbone(resume_inputs)["pooled_output"].numpy()

# 3. Geometric Normalization & Multi-Dimensional Cosine Dot Product
jd_norm = jd_vector / np.linalg.norm(jd_vector, axis=-1, keepdims=True)
resume_norm = resume_vectors / np.linalg.norm(resume_vectors, axis=-1, keepdims=True)
base_scores = np.dot(resume_norm, jd_norm.T).flatten()

# 4. Grounding Step: Highly Target-Specific AI Domain Skill Checklist
ai_domain_keywords = ["tensorflow", "keras", "pytorch", "neural", "predictive", "scikit-learn", "dataset", "deep learning"]

tech_leaderboard = []
for label, embedding_score, resume_text in zip(all_candidate_labels, base_scores, test_resumes):
    text_lower = resume_text.lower()
    skill_count = sum(1 for skill in ai_domain_keywords if skill in text_lower)
    
    # Final Hybrid Equation: Core Neural Space Distance + Weighted Domain Matching
    final_score = embedding_score + (skill_count * 0.04)
    tech_leaderboard.append((label, final_score, skill_count))

# Sort the results by total matching strength
tech_leaderboard.sort(key=lambda x: x, reverse=True)

# 5. Display the Final, Clean Leaderboard
print("\n" + "="*80)
print("🏆 ELITE HYBRID ATS ENGINE: TECHNICAL TALENT LEADERBOARD 🏆")
print("="*80)
for rank, (name, score, counts) in enumerate(tech_leaderboard[:8], 1):
    print(f"Rank {rank}: {name:<45} -> Match Rating: {score*100:.2f}% (Tech Stack Matches: {counts})")
print("="*80)



🏆 ELITE HYBRID ATS ENGINE: TECHNICAL TALENT LEADERBOARD 🏆
Rank 1: Candidate 9 (Machine Learning Engineer)       -> Match Rating: 109.65% (Tech Stack Matches: 4)
Rank 2: Candidate 8 (Machine Learning Engineer)       -> Match Rating: 110.27% (Tech Stack Matches: 4)
Rank 3: Candidate 7 (Machine Learning Engineer)       -> Match Rating: 108.14% (Tech Stack Matches: 3)
Rank 4: Candidate 6 (Frontend Developer)              -> Match Rating: 97.78% (Tech Stack Matches: 0)
Rank 5: Candidate 5 (Frontend Developer)              -> Match Rating: 94.81% (Tech Stack Matches: 0)
Rank 6: Candidate 4 (Machine Learning Engineer)       -> Match Rating: 111.93% (Tech Stack Matches: 4)
Rank 7: Candidate 3 (Backend Developer)               -> Match Rating: 94.04% (Tech Stack Matches: 0)
Rank 8: Candidate 20 (Frontend Developer)             -> Match Rating: 91.75% (Tech Stack Matches: 0)


In [12]:
# 1. Store your custom tricky backend description
tricky_backend_jd = (
    "We need a core Backend Engineer to build high-throughput API pipelines connecting our apps to LLMs. "
    "In this role, you will NOT train ML models, clean datasets, or tune hyperparameters. "
    "Instead, you will build scalable microservices (Go/Python) and optimize vector database retrieval. "
    "Candidates with backgrounds in academic Data Science or PyTorch model creation will not be considered."
)

# 2. Extract features using our unified master model components
jd_ins = preprocessor(np.array([tricky_backend_jd]))
jd_vec = backbone(jd_ins)["pooled_output"].numpy()

# Note: We reuse the 'resume_vectors' we extracted in the previous cell to keep it fast
jd_norm = jd_vector / np.linalg.norm(jd_vector, axis=-1, keepdims=True)
resume_norm = resume_vectors / np.linalg.norm(resume_vectors, axis=-1, keepdims=True)
base_scores = np.dot(resume_norm, jd_norm.T).flatten()

# 3. Define a strict backend skill checklist to ground this specific role
backend_keywords = ["backend", "api", "microservices", "go", "golang", "database", "sql", "scalable", "pipeline"]

tricky_leaderboard = []
for label, embedding_score, resume_text in zip(all_candidate_labels, base_scores, test_resumes):
    text_lower = resume_text.lower()
    skill_count = sum(1 for skill in backend_keywords if skill in text_lower)
    
    # Combined hybrid equation
    final_score = embedding_score + (skill_count * 0.04)
    tricky_leaderboard.append((label, final_score, skill_count))

# FIX: Correctly sort by the numeric score index (index 1), descending
tricky_leaderboard.sort(key=lambda x: x[1], reverse=True)

# 4. Display the results
print("=== TRICKY BACKEND DESCRIPTION LEADERBOARD ===")
for rank, (name, score, counts) in enumerate(tricky_leaderboard[:8], 1):
    print(f"Rank {rank}: {name:<45} -> Match Rating: {score*100:.2f}% (Backend Stack Matches: {counts})")


=== TRICKY BACKEND DESCRIPTION LEADERBOARD ===
Rank 1: Candidate 16 (Backend Developer)              -> Match Rating: 119.20% (Backend Stack Matches: 6)
Rank 2: Candidate 3 (Backend Developer)               -> Match Rating: 114.04% (Backend Stack Matches: 5)
Rank 3: Candidate 15 (Backend Developer)              -> Match Rating: 112.95% (Backend Stack Matches: 4)
Rank 4: Candidate 13 (Backend Developer)              -> Match Rating: 111.60% (Backend Stack Matches: 4)
Rank 5: Candidate 11 (Backend Developer)              -> Match Rating: 111.32% (Backend Stack Matches: 4)
Rank 6: Candidate 14 (Frontend Developer)             -> Match Rating: 108.70% (Backend Stack Matches: 3)
Rank 7: Candidate 6 (Frontend Developer)              -> Match Rating: 105.78% (Backend Stack Matches: 2)
Rank 8: Candidate 9 (Machine Learning Engineer)       -> Match Rating: 105.65% (Backend Stack Matches: 3)


In [6]:
# 1. Define a single consolidated model path
model_save_path = "tech_ats_transformer.keras"

print("💾 Exporting full unified Transformer pipeline to disk...")

# 2. Save the master model (this auto-bundles the preprocessor + backbone + configs)
master_model.save(model_save_path)

print("✅ Complete model architecture and weights saved successfully!")
print(f"-> Saved File: {model_save_path}")


💾 Exporting full unified Transformer pipeline to disk...
✅ Complete model architecture and weights saved successfully!
-> Saved File: tech_ats_transformer.keras


In [7]:
import keras

# Provide the absolute path to Kaggle's working directory folder
model_path = "/kaggle/working/tech_ats_transformer.keras"

print("🔍 Searching for your saved model file...")

# Load everything back with the correct path
engine = keras.models.load_model(model_path)

# Access components seamlessly from the loaded artifact
preprocessor = engine.preprocessor
backbone = engine.backbone

print("🎉 Model successfully loaded from disk!")


🔍 Searching for your saved model file...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/serialization_lib.py:749: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)


🎉 Model successfully loaded from disk!


In [8]:
%%writefile app.py
import streamlit as st
import keras
import numpy as np
import json
import pandas as pd

# 1. PAGE TITLE & APP CONFIGURATION
st.set_page_config(page_title="AI Tech ATS Engine", layout="wide")
st.title("📄 AI Hybrid Technical ATS Screening Engine")
st.write("Evaluate engineering talent using your custom saved KerasHub BERT Transformer architecture.")

# 2. CACHE THE SAVED MODEL LOAD FOR MAXIMUM PERFORMANCE
@st.cache_resource
def load_compiled_engine():
    # Load your saved .keras artifact file cleanly
    engine = keras.models.load_model("tech_ats_transformer.keras")
    return engine.preprocessor, engine.backbone

try:
    preprocessor, backbone = load_compiled_engine()
    st.success("✅ Saved Transformer Model Loaded Successfully from Disk!")
except Exception as e:
    st.error(f"⚠️ Could not load 'tech_ats_transformer.keras'. Ensure the file exists in this directory. Error: {e}")

# 3. INTERACTIVE LAYOUT SCRIPT
col1, col2 = st.columns(2)

with col1:
    st.subheader("🎯 Step 1: Target Vacancy Details")
    job_desc = st.text_area(
        "Enter Job Requirements:",
        "We need a core Backend Engineer to build high-throughput API pipelines connecting our apps to LLMs. "
        "In this role, you will NOT train ML models, clean datasets, or tune hyperparameters. "
        "Instead, you will build scalable microservices (Go/Python) and optimize vector database retrieval.",
        height=180
    )

with col2:
    st.subheader("👥 Step 2: Add Candidate Profiles")
    st.write("Input the mock text profiles to evaluate through the hybrid ranking pipeline.")
    
    cand_1_text = st.text_area("Candidate 1 Profile Text (IT/Backend Focus):", 
                               "Backend Developer with 3 years experience building REST APIs, managing SQL databases, and deploying microservices in Go.", height=60)
    cand_2_text = st.text_area("Candidate 2 Profile Text (AI/ML Focus):", 
                               "Machine Learning Engineer specialized in deep learning, tuning hyperparameters, and training neural networks using PyTorch and TensorFlow.", height=60)

# 4. EXECUTE MATCH ENGINE BUTTON
if st.button("Execute Semantic Ranking Run", type="primary"):
    resumes = [cand_1_text, cand_2_text]
    names = ["Candidate 1 (Backend Focus)", "Candidate 2 (AI/ML Focus)"]
    
    # Run the compiled Preprocessor and Backbone
    jd_inputs = preprocessor(np.array([job_desc]))
    jd_vector = backbone(jd_inputs)["pooled_output"].numpy()
    
    resume_inputs = preprocessor(np.array(resumes))
    resume_vectors = backbone(resume_inputs)["pooled_output"].numpy()
    
    # Calculate geometric cosine similarities
    jd_norm = jd_vector / np.linalg.norm(jd_vector, axis=-1, keepdims=True)
    res_norm = resume_vectors / np.linalg.norm(resume_vectors, axis=-1, keepdims=True)
    base_scores = np.dot(res_norm, jd_norm.T).flatten()
    
    # Strict fallback technical keyword grounding rules
    backend_keywords = ["backend", "api", "microservices", "go", "golang", "database", "sql", "scalable", "pipeline"]
    
    leaderboard = []
    for name, base_score, text in zip(names, base_scores, resumes):
        text_lower = text.lower()
        skills = sum(1 for skill in backend_keywords if skill in text_lower)
        
        # Combine score math and safely cap at a perfect 1.0 (100%) visual limit
        total_score = min(1.0, float(base_score) + (skills * 0.04))
        leaderboard.append((name, total_score, skills))
        
    # Sort numeric score descending
    leaderboard.sort(key=lambda x: x[1], reverse=True)
    
    # 5. RENDER LEADERBOARD UI ELEMENTS
    st.subheader("🏆 Automated Match Rankings")
    for rank, (name, score, count) in enumerate(leaderboard, 1):
        with st.container():
            st.markdown(f"### Rank {rank}: {name}")
            st.write(f"**Verified Technical Target Matches:** {count}")
            st.progress(score)
            st.write(f"**Match Relevance Rating:** {score * 100:.2f}%")
            st.divider()


Writing app.py


In [9]:
import zipfile
from IPython.display import FileLink

print("🤐 Compressing the large model into a zip file to optimize download speed...")

# 1. Compress the 400MB model into a standard zip archive
with zipfile.ZipFile("tech_ats_transformer.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("tech_ats_transformer.keras")

print("🔗 CLICK THE LINK BELOW TO DOWNLOAD:")
# 2. Generate a direct local server link
FileLink(r'tech_ats_transformer.zip')


🤐 Compressing the large model into a zip file to optimize download speed...
🔗 CLICK THE LINK BELOW TO DOWNLOAD:


/kaggle/working/tech_ats_transformer.zip